<a href="https://colab.research.google.com/github/kjfcvx12/Colab/blob/main/06_11_bert_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers datasets evaluate accelerate scikit-learn konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 40.4 MB/s eta 0:00:00


In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm
import time

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"현재 연산 장치: {device}")

현재 연산 장치: cuda


In [5]:
# 1. 데이터 다운로드
raw_dataset = load_dataset("klue/klue", "ynat")

print("데이터 로드 성공!")
print(raw_dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/22.5k [00:00<?, ?B/s]

ynat/train-00000-of-00001.parquet:   0%|          | 0.00/4.17M [00:00<?, ?B/s]

ynat/validation-00000-of-00001.parquet:   0%|          | 0.00/847k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45678 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/9107 [00:00<?, ? examples/s]

데이터 로드 성공!
DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 9107
    })
})


In [6]:
# 2. 데이터프레임 변환
train_df = pd.DataFrame(raw_dataset['train'])
val_df = pd.DataFrame(raw_dataset['validation'])

In [7]:
# 3. 결측치 확인

In [8]:
train_df.isna().sum()

,0
guid,0
title,0
label,0
url,0
date,0


In [9]:
val_df.isna().sum()

,0
guid,0
title,0
label,0
url,0
date,0


In [10]:
# 결측치 제거 후 인덱스 초기화
train_df.dropna(subset=['title'], inplace=True)
val_df.dropna(subset=['title'], inplace=True)

print(f"학습 뉴스 개수: {train_df.shape[0]}개")
print(f"검증 뉴스 개수: {val_df.shape[0]}개")

학습 뉴스 개수: 45678개
검증 뉴스 개수: 9107개


In [11]:
# 4. 샘플링
train_sample = train_df.sample(n=15000, random_state=1)
val_sample = val_df.sample(n=5000, random_state=1)

In [12]:
# 5. 인덱스 초기화
train_reindex = train_sample.reset_index(drop=True)
val_reindex = val_sample.reset_index(drop=True)

In [13]:
# 6. 분류할 분야 세팅
class_names = ['IT과학', '경제', '사회', '생활문화', '세계', '스포츠', '정치']

In [14]:
# 7. 커스텀 데이터셋 정의
class NewsDataset(Dataset):
    def __init__(self, titles, labels):
        # tolist()로 표형태의 데이터셋을 리스트 변환
        # 인덱싱 속도 극대화
        self.titles = titles.tolist()
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.titles[idx], self.labels[idx]

In [15]:
# 8. 데이터셋 객체 생성
train_dataset = NewsDataset(train_df['title'], train_df['label'])
val_dataset = NewsDataset(val_df['title'], val_df['label'])

In [16]:
# 9. 데이터로더 생성
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [17]:
# 10. 반복자 생성
data_iterate = iter(train_loader)
document_batch, label_batch = next(data_iterate)

In [18]:
# 11. 첫 배치 32개 중 10개만 출력
for i in range(10):
    text = document_batch[i]
    label = label_batch[i].item()
    print(f'리뷰: {text}')
    print(f'label: {label} ({class_names[label]})')
    print("-" * 60)

리뷰: 듀랜트·커리 55점 합작…골든스테이트 클리블랜드 원정서 완승
label: 5 (스포츠)
------------------------------------------------------------
리뷰: 프로배구 옵션 상한액 도입에 7개 구단 찬성…과제는
label: 5 (스포츠)
------------------------------------------------------------
리뷰: 메리츠증권 코웨이 적극적 주주환원 정책 주목해야
label: 1 (경제)
------------------------------------------------------------
리뷰: 카카오 본사·자회사 협의 기구 신설…송지호 대표가 수장
label: 2 (사회)
------------------------------------------------------------
리뷰: 한미 대규모 연합훈련 시작…北 핵심시설 타격 초점
label: 6 (정치)
------------------------------------------------------------
리뷰: 김무성 朴대통령 얘기 않겠다…질문하지 마라
label: 6 (정치)
------------------------------------------------------------
리뷰: 北 연평도포격도발 승리 주장하며 경축행사
label: 2 (사회)
------------------------------------------------------------
리뷰: 삼성바이오 논란 금융당국 책임 국민청원 잇따라종합
label: 1 (경제)
------------------------------------------------------------
리뷰: AFC컵 조별리그 이라크내 두 경기 4월로 연기
label: 5 (스포츠)
------------------------------------------------------------
리뷰: 이라크 주변국 입법기관 회의 열어…중재자 역할 부각
la

In [19]:
# 12. 모델 다운로드
model_id = 'bert-base-multilingual-cased'
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [20]:
# 13. bert 모델 작성
class BertModel(nn.Module):
    def __init__(self, model_id):
        super().__init__()
        # bert-base-multilingual-cased 다국어 모델 세팅
        self.bert = AutoModel.from_pretrained(model_id)
        # 10% 드롭아웃 과적합 방지
        self.dropout = nn.Dropout(0.1)
        # 뉴스 카테고리가 7개
        # 출력도 7차원
        self.classifier = nn.Linear(self.bert.config.hidden_size, 7)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # 토큰 조각 병합
        pooled_output = outputs.pooler_output

        # 드롭아웃 과적합 방지
        net = self.dropout(pooled_output)
        # 분야 7개 정제
        logits = self.classifier(net)

        return logits

In [21]:
# 14. 작성한 모델에 bert-base-multilingual-cased 적용
model = BertModel(model_id)
model.to(device)

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine

In [22]:
# 15. 모델 훈련&검증
epochs = 2
optimizer = optim.AdamW(model.parameters(), lr=3e-5)
# 다중 분류 표준 CrossEntropyLoss
criterion = nn.CrossEntropyLoss()

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

print(f'Training model with {model_id} on {device}\n')

for epoch in range(epochs):
    # 훈련 시작
    model.train()
    train_loss, train_correct = 0, 0
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")

    for batch in train_bar:
        texts, labels = batch

        # 실시간 토크나이징&GPU 이동
        encoded_input = tokenizer(
            list(texts),
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        # 문자열 리스트변환
        # 부족한 부분 추가
        # 넘치는 부분 제거
        # 최대길이 128
        # 파이토치 반환

        # 다중 분류 정수 사용
        labels = labels.to(device).long()

        # 예전 기울기 리셋
        optimizer.zero_grad()

        # 순전파 진행
        logits = model(**encoded_input)
        loss = criterion(logits, labels)

        # 역전파(오차 원인 분석)
        loss.backward()
        # 기울기 이동
        optimizer.step()

        train_loss += loss.item()

        # 최대값 최종 예측값
        preds = torch.argmax(logits, dim=1)
        # 정답일 경우 추가
        train_correct += (preds == labels).sum().item()

        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    # 검증 시작
    model.eval()
    val_loss, val_correct = 0, 0

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
            texts, labels = batch

            encoded_input = tokenizer(
                list(texts),
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(device)

            labels = labels.to(device).long()

            logits = model(**encoded_input)
            loss = criterion(logits, labels)

            val_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            val_correct += (preds == labels).sum().item()

    # 성적 평균 산출
    epoch_train_loss = train_loss / len(train_loader)
    epoch_train_acc = train_correct / len(train_dataset)
    epoch_val_loss = val_loss / len(test_loader)
    epoch_val_acc = val_correct / len(val_dataset)

    history['loss'].append(epoch_train_loss)
    history['accuracy'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_accuracy'].append(epoch_val_acc)

    print(f"Epoch {epoch+1} 결과")
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% | Val Acc: {epoch_val_acc*100:.2f}%\n")


Training model with bert-base-multilingual-cased on cuda



Epoch 1/2 [Train]:   0%|          | 0/1428 [00:00<?, ?it/s]

Epoch 1/2 [Val]:   0%|          | 0/285 [00:00<?, ?it/s]

Epoch 1 결과
Train Loss: 0.5663 | Train Acc: 81.23% | Val Acc: 80.88%



Epoch 2/2 [Train]:   0%|          | 0/1428 [00:00<?, ?it/s]

Epoch 2/2 [Val]:   0%|          | 0/285 [00:00<?, ?it/s]

Epoch 2 결과
Train Loss: 0.3820 | Train Acc: 86.80% | Val Acc: 79.35%



In [23]:
# 16. 결과 확인
data_iterate = iter(train_loader)
document_batch, label_batch = next(data_iterate)

print("기사 분류\n")

for i in range(10):
    text = document_batch[i]
    label = label_batch[i].item()

    print(f'뉴스 제목: {text}')
    print(f'카테고리: {label} ({class_names[label]})')
    print("-" * 50)

기사 분류

뉴스 제목: 고가아파트 취득에 세무조사 칼 빼들어… 257명 세무조사 착수
카테고리: 1 (경제)
--------------------------------------------------
뉴스 제목: 최고위원회의에서 발언하는 이해찬
카테고리: 6 (정치)
--------------------------------------------------
뉴스 제목: KT 2021년까지 5대 플랫폼 매출 비중 30%로 확대
카테고리: 1 (경제)
--------------------------------------------------
뉴스 제목: 朴대통령 안보·국익 따라 사드 검토…전술핵 생각안해
카테고리: 6 (정치)
--------------------------------------------------
뉴스 제목: 김성태 MB를 법정에 세우려는 정치보복…정치한풀이 정권
카테고리: 6 (정치)
--------------------------------------------------
뉴스 제목: 일제강제징용 유가족 청구권 자금 환수 촉구
카테고리: 2 (사회)
--------------------------------------------------
뉴스 제목: 폴란드서 이란 겨냥 美주도 60여개국 중동문제회의 시작
카테고리: 4 (세계)
--------------------------------------------------
뉴스 제목: 정운찬 총재 인센티브는 산업화 기초 세우기 위해
카테고리: 2 (사회)
--------------------------------------------------
뉴스 제목: 낯선 실험 vs 치밀한 서사…1980년대생 작가들 새 소설집
카테고리: 3 (생활문화)
--------------------------------------------------
뉴스 제목: 아시아 통화 강세에 연동…원달러 환율 5.6원 하락
카테고리: 1 (경제)
------------------------------------------

In [24]:
import torch.nn.functional as F
import numpy as np

my_news_examples = [
    "나사, 화성에서 고대 생명체 흔적 발견... 전 세계 우주 과학계 들썩",
    "한국은행, 경기 부양 위해 기준금리 연 2.0%로 전격 인하 결정",
    "경찰, 대규모 전세사기 일당 일제 검거... 피해액만 수백억 원 달해",
    "파리 패션위크 개막... 올가을 전 세계를 사로잡을 메인 트렌드는?",
    "백악관 “북한의 미사일 도발, 강력 규제 조치 논의 중” 공식 발표",
    "손흥민, 멀티골 폭발하며 토트넘 극적인 역전승 견인... 평점 9.5만점"
]

model.eval()

encoded_test_input = tokenizer(
    my_news_examples,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    logits = model(**encoded_test_input)
    preds = torch.argmax(logits, dim=1).cpu().numpy()

for i in range(len(my_news_examples)):
    title = my_news_examples[i]
    best_label_idx = preds[i]

    print(f"뉴스 제목: {title:<50}")
    print(f"판단 결과 : {class_names[best_label_idx]} 분야")
    print("-" * 65)


뉴스 제목: 나사, 화성에서 고대 생명체 흔적 발견... 전 세계 우주 과학계 들썩           
판단 결과 : IT과학 분야
-----------------------------------------------------------------
뉴스 제목: 한국은행, 경기 부양 위해 기준금리 연 2.0%로 전격 인하 결정              
판단 결과 : 경제 분야
-----------------------------------------------------------------
뉴스 제목: 경찰, 대규모 전세사기 일당 일제 검거... 피해액만 수백억 원 달해            
판단 결과 : 사회 분야
-----------------------------------------------------------------
뉴스 제목: 파리 패션위크 개막... 올가을 전 세계를 사로잡을 메인 트렌드는?             
판단 결과 : 생활문화 분야
-----------------------------------------------------------------
뉴스 제목: 백악관 “북한의 미사일 도발, 강력 규제 조치 논의 중” 공식 발표             
판단 결과 : 정치 분야
-----------------------------------------------------------------
뉴스 제목: 손흥민, 멀티골 폭발하며 토트넘 극적인 역전승 견인... 평점 9.5만점          
판단 결과 : 스포츠 분야
-----------------------------------------------------------------
